<a href="https://colab.research.google.com/github/rdelhibabu/DQI_QFL/blob/main/DQI_QFL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pennylane==0.35.0 torch==2.2.0 scikit-learn

In [2]:
!pip install pennylane==0.35.0 torch==2.2.0 scikit-learn autoray==0.6.12

In [3]:
!pip install pennylane==0.35.0 torch==2.2.0 scikit-learn autoray==0.6.12 "numpy<2"

In [ ]:
"""
Cross-Silo Quantum Federated Learning (CS-QFL) Simulation
Author: Radhakrishnan Delhibabu (rdelhibabu@vit.ac.in)
Institution: Vellore Institute of Technology (VIT), School of Computer Science and Engineering
Description:
Simulates a 5-node decentralized quantum federated learning architecture using
the Breast Cancer Wisconsin dataset partitioned via a Dirichlet distribution.
Utilizes PennyLane and PyTorch for Variational Quantum Circuit (VQC) training.
"""

import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# ==========================================
# 1. Hyperparameters & Configuration (Table 5)
# ==========================================
NUM_QUBITS = 8
NUM_LAYERS = 4
NUM_SILOS = 5
GLOBAL_ROUNDS = 50
LOCAL_EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 0.01
LR_DECAY = 0.99
DIRICHLET_ALPHA = 0.5
NOISE_PROB_LOCAL = 0.005   # Local depolarization noise
NOISE_PROB_CHANNEL = 0.02  # Channel phase damping noise

# Fix seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

# ==========================================
# 2. Data Preparation & Non-IID Partitioning
# ==========================================
def prepare_and_partition_data():
    """Loads Breast Cancer dataset, applies PCA, scales, and partitions (Table 4)."""
    data = load_breast_cancer()
    X, y = data.data, data.target

    # PCA down to 8 dimensions for 8-qubit angle encoding
    pca = PCA(n_components=NUM_QUBITS)
    X_pca = pca.fit_transform(X)

    # Scale features to [-pi, pi] for quantum angle encoding
    scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
    X_scaled = scaler.fit_transform(X_pca)

    # Dirichlet Partitioning to simulate Non-IID silos
    silo_indices = [[] for _ in range(NUM_SILOS)]
    classes = np.unique(y)

    for c in classes:
        idx_c = np.where(y == c)[0]
        np.random.shuffle(idx_c)
        proportions = np.random.dirichlet(np.repeat(DIRICHLET_ALPHA, NUM_SILOS))
        proportions = np.array([p * len(idx_c) for p in proportions]).astype(int)

        # Adjust for rounding errors
        proportions[-1] = len(idx_c) - sum(proportions[:-1])

        split_indices = np.split(idx_c, np.cumsum(proportions)[:-1])
        for i in range(NUM_SILOS):
            silo_indices[i].extend(split_indices[i])

    silo_datasets = []
    for indices in silo_indices:
        np.random.shuffle(indices)
        X_silo, y_silo = X_scaled[indices], y[indices]
        # 80/20 train/val split per silo
        X_train, X_val, y_train, y_val = train_test_split(X_silo, y_silo, test_size=0.2, random_state=42)
        silo_datasets.append({
            'train': (torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)),
            'val': (torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))
        })

    return silo_datasets, (torch.tensor(X_scaled, dtype=torch.float32), torch.tensor(y, dtype=torch.float32))

# ==========================================
# 3. Quantum Circuit Definition (HEA)
# ==========================================
dev = qml.device("default.mixed", wires=NUM_QUBITS)

@qml.qnode(dev, interface="torch", diff_method="parameter-shift")
def vqc_node(inputs, weights):
    """
    Local VQC with Angle Encoding and Hardware-Efficient Ansatz.
    weights shape: (NUM_LAYERS, NUM_QUBITS, 2)
    """
    # 1. State Encoding (Eq. 2)
    for i in range(NUM_QUBITS):
        # BUG FIX: Use [..., i] to properly slice the feature dimension
        # across the entire batch (handles both 1D and 2D tensors)
        qml.RY(inputs[..., i], wires=i)
        qml.RZ(inputs[..., i], wires=i)

    # 2. Hardware-Efficient Ansatz (Eq. 3)
    for layer in range(NUM_LAYERS):
        # Rotation block
        for i in range(NUM_QUBITS):
            qml.RY(weights[layer, i, 0], wires=i)
            qml.RZ(weights[layer, i, 1], wires=i)
        # Entanglement block (CZ gates)
        for i in range(NUM_QUBITS - 1):
            qml.CZ(wires=[i, i + 1])
        qml.CZ(wires=[NUM_QUBITS - 1, 0])

    # Apply local depolarization noise simulation (Hardware abstraction)
    for i in range(NUM_QUBITS):
        qml.DepolarizingChannel(NOISE_PROB_LOCAL, wires=i)

    # 3. Measurement (Expectation value of PauliZ on first qubit)
    return qml.expval(qml.PauliZ(0))

class QuantumClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        weight_shapes = {"weights": (NUM_LAYERS, NUM_QUBITS, 2)}
        # Initialize weights uniformly [0, 2*pi]
        init_method = lambda x: torch.nn.init.uniform_(x, 0, 2 * np.pi)
        self.qlayer = qml.qnn.TorchLayer(vqc_node, weight_shapes, init_method=init_method)

    def forward(self, x):
        # Output is in [-1, 1], map to [0, 1] for binary cross entropy
        out = self.qlayer(x)
        return (out + 1.0) / 2.0
# ==========================================
# 4. Federated Aggregation Mechanics
# ==========================================
def simulate_qspa_channel_noise(params, noise_prob):
    """Simulates phase-damping degradation over the quantum communication channel."""
    noise = torch.randn_like(params) * noise_prob
    return params + noise

def homomorphic_average(local_weights):
    """
    Computes the global average of parameters.
    In the physical implementation, this is done via the QFT adder on encrypted states.
    Here we mathematically simulate the decrypted outcome subject to channel noise.
    """
    avg_weights = torch.mean(torch.stack(local_weights), dim=0)
    return simulate_qspa_channel_noise(avg_weights, NOISE_PROB_CHANNEL)

# ==========================================
# 5. Main CS-QFL Training Loop (Algorithm 4)
# ==========================================
def train_cs_qfl():
    silo_datasets, global_data = prepare_and_partition_data()
    X_global, y_global = global_data

    # Initialize global model
    global_model = QuantumClassifier()
    global_weights = global_model.qlayer.weights.data.clone()

    loss_fn = nn.BCELoss()
    global_accuracies = []

    print(f"--- Starting CS-QFL Simulation over {GLOBAL_ROUNDS} Rounds ---")

    for round_idx in range(GLOBAL_ROUNDS):
        local_weight_updates = []

        # Calculate decaying learning rate
        current_lr = LEARNING_RATE * (LR_DECAY ** round_idx)

        # Phase 1: Local QML Training (in parallel/sequential for simulation)
        for silo_idx in range(NUM_SILOS):
            local_model = QuantumClassifier()
            local_model.qlayer.weights.data = global_weights.clone()
            optimizer = optim.Adam(local_model.parameters(), lr=current_lr)

            X_train, y_train = silo_datasets[silo_idx]['train']

            local_model.train()
            for epoch in range(LOCAL_EPOCHS):
                # Mini-batching
                permutation = torch.randperm(X_train.size()[0])
                for i in range(0, X_train.size()[0], BATCH_SIZE):
                    indices = permutation[i:i+BATCH_SIZE]
                    batch_X, batch_y = X_train[indices], y_train[indices]

                    optimizer.zero_grad()
                    predictions = local_model(batch_X)
                    loss = loss_fn(predictions, batch_y)
                    loss.backward()
                    optimizer.step()

            # Save optimized local parameters
            local_weight_updates.append(local_model.qlayer.weights.data.clone())

        # Phase 2: Secure Aggregation (Server side simulation)
        global_weights = homomorphic_average(local_weight_updates)

        # Phase 3: Global Model Evaluation
        global_model.qlayer.weights.data = global_weights.clone()
        global_model.eval()
        with torch.no_grad():
            preds = global_model(X_global)
            predicted_classes = (preds > 0.5).float()
            accuracy = (predicted_classes == y_global).float().mean().item()
            global_accuracies.append(accuracy)

        print(f"Round {round_idx + 1}/{GLOBAL_ROUNDS} | Global Validation Accuracy: {accuracy * 100:.2f}%")

    print("\n--- Simulation Complete ---")
    print(f"Final CS-QFL Global Accuracy: {global_accuracies[-1] * 100:.2f}%")
    return global_accuracies

if __name__ == "__main__":
    accuracies = train_cs_qfl()

--- Starting CS-QFL Simulation over 50 Rounds ---
